# 05 — Data Integrity (Workstreams D+E): reproduce the analysis & plots

Interactive companion to **`docs/data-integrity-findings.md`**. It **loads the canonical results**
(`data/data_integrity_results.json`, produced by the tested `scripts/run_data_integrity.py`), shows
every reported table, and **redraws the plots live** — and offers an opt-in cell to recompute the
whole report from scratch. It does **not** re-implement the analysis. (For what each statistic means,
see `docs/metrics-glossary.md`.)

## How this differs from `04_ssl_vs_prosody.ipynb`

| | `04` (Phase 0.5) | `05` (this notebook, D+E) |
|---|---|---|
| **Data unit** | 55 radio **clips**, each windowed into many segments (pseudoreplicated) | **178 independent segments, one per recording**, fixed 30 s (no re-windowing) |
| **Coverage** | very uneven (italian = 1 clip) | balanced (25/lang; greek 15, spanish 14) |
| **Compute** | re-embeds + re-cleans **inline** with its own copies of the logic; sweeps length×layer | **loads the canonical results** + redraws plots live via the **unit-tested** pipeline; opt-in full recompute |
| **Config** | best-of-15 (3 lengths × 5 layers) | fixed headline **layer 16** + a cheap {12,16,last} sensitivity check |
| **Aggregation** | mean + std | **robust median/MAD**, **per-channel-weighted** centroids |
| **Robustness** | within-recording stability only | **bootstrap CIs**, **leave-one-station-out**, **outlier detection (report-both)** |
| **Confound** | stations mostly singletons → degenerate (NaN) | **~3 segments/channel → a real measured effect** |
| **Framing** | establishes the baseline | **before/after** vs the Phase-0.5 confound |

In short: `04` *did* the exploratory compute; `05` is a thin, fast **reproduction/exploration** layer
over the hardened, unit-tested D+E pipeline.

In [ ]:
import importlib.util
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram

import musiclang
from musiclang.config import REPO_ROOT, DATA_DIR   # absolute paths (independent of the notebook's CWD)

# scripts/ is not an importable package, so load the tested driver as a module by file path
# (same convention as tests/test_run_data_integrity.py). We then call its real functions —
# no analysis logic is re-implemented in this notebook.
_spec = importlib.util.spec_from_file_location(
    "run_data_integrity", REPO_ROOT / "scripts" / "run_data_integrity.py"
)
rdi = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rdi)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("loaded", REPO_ROOT / "scripts" / "run_data_integrity.py")

## 1. Inputs & coverage

Read the cached feature tables produced by `scripts/build_segment_prosody.py` (prosody + provenance)
and `scripts/embed_segments.py` (the XLS-R embedding caches). Note the dimensionalities: prosody = 16
hand-engineered scalars; SSL = **1024** per segment — and that width is the model's hidden size, the
**same at every layer** (a "layer" is transformer *depth*, not width; see §6).

> All of `data/` is gitignored, so these files must already exist locally (built by Workstreams
> A–C + the D+E feature scripts). On a fresh checkout, build them first.

In [ ]:
prosody_df = pd.read_parquet(DATA_DIR / "segment_features_prosody.parquet")
prov_df    = pd.read_parquet(DATA_DIR / "segment_provenance.parquet")
emb = {
    12:     pd.read_parquet(DATA_DIR / "segment_embeddings_xlsr_l12.parquet"),
    16:     pd.read_parquet(DATA_DIR / "segment_embeddings_xlsr_l16.parquet"),
    "last": pd.read_parquet(DATA_DIR / "segment_embeddings_xlsr_llast.parquet"),
}
n_emb = len([c for c in emb[16].columns if c.startswith("emb_")])
print(f"{len(prosody_df)} segments  |  prosody dims = {prosody_df.shape[1]}  |  "
      f"SSL dims = {n_emb} (same at every layer)  |  {prov_df['channel_id'].nunique()} channels")

coverage = (prov_df.groupby("language")
            .agg(segments=("channel_id", "size"), channels=("channel_id", "nunique"))
            .sort_values("segments", ascending=False))
coverage

## 2. Load the canonical results

`data/data_integrity_results.json` is the exact artifact the findings doc cites (Mantel at 10 000
permutations, bootstrap `n_boot = 1000`). Loading it is instant and keeps this notebook in lockstep
with the report.

> **Why load instead of recompute here?** The SSL bootstrap rebuilds a per-language centroid over
> 1024-dim embeddings thousands of times — a full recompute is ~15–60 min. So we load by default and
> redraw the (cheap) plots live; **§8 has an opt-in cell to recompute from scratch.** If the JSON is
> missing, run `uv run python scripts/run_data_integrity.py` (or the §8 cell) first.

In [ ]:
RESULTS_PATH = DATA_DIR / "data_integrity_results.json"
if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"{RESULTS_PATH} not found (data/ is gitignored). Run "
        "`uv run python scripts/run_data_integrity.py` first, or set RECOMPUTE=True in the §8 cell."
    )
with open(RESULTS_PATH, encoding="utf-8") as f:
    rep = json.load(f)
print("loaded canonical results:", [k for k in rep])

## 3. The confound re-test (the central question)

Does the geometry cluster by **language** or by **station/channel**? The SSL cosine geometry is the
apples-to-apples comparison to Phase 0.5 (the `phase05` reference gaps are carried for the SSL method
only — they are not comparable to the prosody euclidean space).

In [ ]:
c = rep["ssl"]["confound"]
confound_tbl = pd.DataFrame([
    {"geometry": "Phase 0.5 (55-clip)", "language_gap": c["phase05"]["language_gap"],
     "station_gap": c["phase05"]["station_gap"], "language_sil": np.nan, "station_sil": np.nan},
    {"geometry": "D+E (178 indep.)", "language_gap": c["language_gap"], "station_gap": c["station_gap"],
     "language_sil": c["language_silhouette"], "station_sil": c["station_silhouette"]},
])
confound_tbl["station/lang ratio"] = confound_tbl["station_gap"] / confound_tbl["language_gap"]
print("SSL (XLS-R layer 16, cosine) — headline before/after:")
display(confound_tbl)

cg = rep["ssl"]["stability"]["confound_gap_loso"]
print("\nStability — leave-one-station-out over all 59 channels (SSL):")
display(pd.DataFrame(cg).T[["min", "median", "max"]])

pc = rep["prosody"]["confound"]
print(f"\nProsody (standardized euclidean, own terms — NOT comparable to Phase 0.5): "
      f"language_gap={pc['language_gap']:.3f}  station_gap={pc['station_gap']:.3f}  "
      f"(ratio {pc['station_gap']/pc['language_gap']:.1f}x)  |  "
      f"language_sil={pc['language_silhouette']:.3f}  station_sil={pc['station_silhouette']:.3f}")

## 4. Method comparison (bootstrap CIs + LOSO)

Per-language proximity → rhythm-class **silhouette** and the genealogical **Mantel r**, each with a
95 % bootstrap CI and a leave-one-station-out median. A CI that **excludes 0** is the meaningful signal
(the SSL win); the prosody CIs straddle 0. The Mantel *permutation* p is separate from the bootstrap
CI — they test different nulls (glossary §F3).

In [ ]:
def method_row(name, r):
    m, s = r["metrics"]["full"], r["stability"]
    return {
        "method": name,
        "rhythm_sil": m["rhythm_silhouette"],
        "sil_CI_lo": s["bootstrap_ci"]["lo"], "sil_CI_hi": s["bootstrap_ci"]["hi"],
        "sil_LOSO_med": s["leave_one_station_out"]["median"],
        "mantel_r": m["mantel_r"], "mantel_p": m["mantel_p"],
        "mantel_CI_lo": s["mantel_r"]["bootstrap_ci"]["lo"], "mantel_CI_hi": s["mantel_r"]["bootstrap_ci"]["hi"],
        "mantel_LOSO_med": s["mantel_r"]["leave_one_station_out"]["median"],
    }

pd.DataFrame([method_row("prosody", rep["prosody"]), method_row("XLS-R (L16)", rep["ssl"])]).set_index("method")

## 5. Outlier strand (report-both)

Outliers are **flagged, then results recomputed with and without exclusion** — never silently
filtered. Watch how aggressively `IsolationForest(contamination="auto")` flags the prosody space, and
how excluding those segments *degrades* the prosody metrics (robust median aggregation already handles
anomalies). The per-segment flags live in `data/segment_outliers.parquet`.

In [ ]:
outliers = pd.read_parquet(DATA_DIR / "segment_outliers.parquet")
print("Outlier flags (of 178):")
display(outliers.groupby(["space", "detector"])["is_outlier"].sum().to_frame("flagged"))

def report_both(name, r):
    f, e, d = r["metrics"]["full"], r["metrics"]["excluded"], r["metrics"]["delta"]
    return {"method": name,
            "sil full": f["rhythm_silhouette"], "sil excl": e["rhythm_silhouette"], "sil Δ": d["rhythm_silhouette"],
            "mantel full": f["mantel_r"], "mantel excl": e["mantel_r"], "mantel Δ": d["mantel_r"],
            "n excluded": r["outlier_counts"]["union"]}

print("\nReport-both (full vs outlier-excluded):")
pd.DataFrame([report_both("prosody", rep["prosody"]), report_both("XLS-R (L16)", rep["ssl"])]).set_index("method")

## 6. Layer sensitivity

The same confound + rhythm-silhouette scores read from XLS-R layers {12, 16, last}. **"Layer 16" is
transformer depth, not width** — every layer is 1024-dim; the index selects how much of the network's
processing has been applied. Mid layers carry more phonetic/prosodic substrate; the last layer drifts
to the pretraining objective and collapses here.

In [ ]:
ls = rep["layer_sensitivity"]
pd.DataFrame([
    {"layer": k,
     "language_sil": ls[k]["confound"]["language_silhouette"],
     "station_sil":  ls[k]["confound"]["station_silhouette"],
     "language_gap": ls[k]["confound"]["language_gap"],
     "station_gap":  ls[k]["confound"]["station_gap"],
     "rhythm_sil":   ls[k]["rhythm_silhouette"]}
    for k in ("12", "16", "last")
]).set_index("layer")

## 7. Figures (redrawn live)

Same construction the committed figures use (tested `proximity_pipeline` → `linkage_matrix` /
`mds_2d`) — fast, so we recompute them here rather than loading PNGs. **The MDS axes are synthesized
coordinates, not feature dimensions** — read neighbours and clusters, not axis values; the whole map
can be rotated/reflected freely.

In [ ]:
RHYTHM_COLORS = {"stress": "tab:red", "syllable": "tab:blue", "intermediate": "tab:green"}

def plot_language(method, feat_df):
    dist = rdi.proximity_pipeline(feat_df, prov_df, method=method)   # tested per-language proximity
    langs = list(dist.index)
    fig, (axd, axm) = plt.subplots(1, 2, figsize=(13, 4.6))
    dendrogram(rdi.linkage_matrix(dist), labels=langs, ax=axd)
    axd.set_title(f"{method}: language dendrogram (Ward linkage)")
    axd.tick_params(axis="x", rotation=45)
    coords = rdi.mds_2d(dist)
    for lang in langs:
        cls = rdi.RHYTHM_CLASS.get(lang, "intermediate")
        axm.scatter(coords.loc[lang, "mds_x"], coords.loc[lang, "mds_y"],
                    color=RHYTHM_COLORS.get(cls, "gray"), s=90)
        axm.annotate(f"{lang}\n{'/'.join(rdi.LINEAGE.get(lang, [lang]))}",
                     (coords.loc[lang, "mds_x"], coords.loc[lang, "mds_y"]),
                     fontsize=7, xytext=(4, 4), textcoords="offset points")
    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=col, markersize=9, label=cls)
               for cls, col in RHYTHM_COLORS.items()]
    axm.legend(handles=handles, title="rhythm class", fontsize=8)
    axm.set_title(f"{method}: language MDS  (axes = synthesized coords, not features)")
    plt.tight_layout()
    plt.show()

plot_language("prosody", prosody_df)
plot_language("ssl", emb[16])

### Segment-level confound picture (the visual story)

178 segments in one MDS, coloured by **language** (left) and by **channel** (right). If segments group
by language more than by channel, the confound is under control.

In [ ]:
seg_dist = rdi.distance_matrix(rdi._l2_normalize(emb[16]), metric="cosine")
seg_coords = rdi.mds_2d(seg_dist)
fig, (ax_l, ax_c) = plt.subplots(1, 2, figsize=(13, 5))
for lang, grp in prov_df.groupby("language"):
    ids = [i for i in grp.index if i in seg_coords.index]
    ax_l.scatter(seg_coords.loc[ids, "mds_x"], seg_coords.loc[ids, "mds_y"], label=lang, s=20)
ax_l.set_title("segment MDS (SSL) — coloured by language")
ax_l.legend(fontsize=6, ncol=2)
for chan, grp in prov_df.groupby("channel_id"):
    ids = [i for i in grp.index if i in seg_coords.index]
    ax_c.scatter(seg_coords.loc[ids, "mds_x"], seg_coords.loc[ids, "mds_y"], s=20)
ax_c.set_title("segment MDS (SSL) — coloured by channel (59 channels, no legend)")
plt.tight_layout()
plt.show()

## 8. Recompute from scratch (opt-in) & regenerate artifacts

Everything above **loads** the canonical results and redraws plots live. To actually *run the full
analysis* — the same computation `main()` performs — set `RECOMPUTE = True` below. This calls the
tested pipeline end-to-end and overwrites `data/data_integrity_results.json`,
`data/segment_outliers.parquet`, and `docs/figures/data-integrity/*.png`.

> **Slow.** The SSL bootstrap dominates: `N_BOOT = 1000` (report parity) is ~15–60 min; drop `N_BOOT`
> for a faster, slightly rougher CI. Only `N_BOOT` orchestration lives here — every metric reuses the
> tested `run_data_integrity` / `musiclang.validation` functions. Equivalent one-liner from a shell:
> `uv run python scripts/run_data_integrity.py`.

In [ ]:
RECOMPUTE = False    # <- set True to run the full analysis from scratch (slow)
N_BOOT = 1000        # committed results use 1000; lower = faster but rougher CIs

if RECOMPUTE:
    def _stability(feat_df, method, n_boot=N_BOOT):
        sil_los  = rdi.leave_one_station_out(feat_df, prov_df, rdi._rhythm_silhouette_fn, method=method)
        sil_boot = rdi.bootstrap_metric_ci(feat_df, prov_df, rdi._rhythm_silhouette_fn, method=method, n_boot=n_boot, seed=0)
        man_los  = rdi.leave_one_station_out(feat_df, prov_df, rdi._mantel_r_fn, method=method)
        man_boot = rdi.bootstrap_metric_ci(feat_df, prov_df, rdi._mantel_r_fn, method=method, n_boot=n_boot, seed=0)
        return {"leave_one_station_out": rdi._loso_spread(sil_los), "bootstrap_ci": sil_boot,
                "mantel_r": {"leave_one_station_out": rdi._loso_spread(man_los), "bootstrap_ci": man_boot}}

    fresh = rdi.assemble_report(prosody_df, emb[16], prov_df, phase05=rdi.PHASE05)
    fresh["layer_sensitivity"]["12"]   = rdi._layer_entry(emb[12],     prov_df, rdi.PHASE05)
    fresh["layer_sensitivity"]["last"] = rdi._layer_entry(emb["last"], prov_df, rdi.PHASE05)
    fresh["prosody"]["stability"] = _stability(prosody_df, "prosody")
    fresh["ssl"]["stability"]     = _stability(emb[16], "ssl")
    fresh["prosody"]["stability"]["confound_gap_loso"] = rdi.confound_gap_loso(prosody_df, prov_df, "prosody", rdi.PHASE05)
    fresh["ssl"]["stability"]["confound_gap_loso"]     = rdi.confound_gap_loso(emb[16],    prov_df, "ssl", rdi.PHASE05)
    rdi.write_results(fresh, results_path=DATA_DIR / "data_integrity_results.json",
                      outliers_path=DATA_DIR / "segment_outliers.parquet")
    rdi.make_figures(prosody_df, emb[16], prov_df, out_dir=REPO_ROOT / "docs" / "figures" / "data-integrity")
    rep = fresh   # refresh the in-notebook report so the tables above reflect the recompute on re-run
    print(f"recomputed (N_BOOT={N_BOOT}) and wrote results + figures")
else:
    print("RECOMPUTE=False — using the loaded canonical results. Set RECOMPUTE=True to regenerate.")

**Read next:** `docs/data-integrity-findings.md` (the written interpretation) and
`docs/metrics-glossary.md` (what every statistic here means).